In [ ]:
# =============================================================================
# PART 1: CORE BLOCKCHAIN INFRASTRUCTURE
# =============================================================================

@dataclass
class Transaction:
    tx_id: str
    sender: str
    operation: str  # 'issue_record', 'update_score', 'logical_delete'
    data_hash: str
    timestamp: float
    signature: str
    metadata: Dict

    def to_dict(self):
        return {
            'tx_id': self.tx_id,
            'sender': self.sender,
            'operation': self.operation,
            'data_hash': self.data_hash,
            'timestamp': self.timestamp,
            'signature': self.signature[:20] + '...',  # Truncated for display
            'metadata': self.metadata
        }

@dataclass
class Block:
    index: int
    timestamp: float
    transactions: List[Transaction]
    previous_hash: str
    merkle_root: str
    hash: str
    validator: str

    def compute_hash(self) -> str:
        block_string = json.dumps({
            'index': self.index,
            'timestamp': self.timestamp,
            'previous_hash': self.previous_hash,
            'merkle_root': self.merkle_root,
            'validator': self.validator
        }, sort_keys=True)
        return hashlib.sha256(block_string.encode()).hexdigest()

class PermissionedBlockchain:
    """
    Simulates the permissioned IBFT-2.0 chain with 25 validators
    (PBoC + Supreme Court + NDRC + SAMR + 21 banks)
    """
    def __init__(self):
        self.chain: List[Block] = []
        self.pending_transactions: List[Transaction] = []
        self.validators = [
            'PBoC', 'Supreme_Court', 'NDRC', 'SAMR',
            'Bank_of_China', 'ICBC', 'CCB', 'ABC', 'BoComm',
            'CMB', 'SPDB', 'CITIC', 'CMBC', 'CIB',
            'CEB', 'HXB', 'BOS', 'NBCB', 'BEA',
            'HSBC_CN', 'StanChart_CN', 'Deutsche_CN', 'BNP_CN', 'JPM_CN', 'GS_CN'
        ]
        self.threshold = 3  # 3-of-5 for critical operations
        self.create_genesis_block()

    def create_genesis_block(self):
        genesis = Block(
            index=0,
            timestamp=time.time(),
            transactions=[],
            previous_hash="0" * 64,
            merkle_root="0" * 64,
            hash="",
            validator="Genesis"
        )
        genesis.hash = genesis.compute_hash()
        self.chain.append(genesis)

    def compute_merkle_root(self, transactions: List[Transaction]) -> str:
        """Compute Merkle root for transaction integrity"""
        if not transactions:
            return hashlib.sha256(b'').hexdigest()

        hashes = [hashlib.sha256(tx.tx_id.encode()).digest() for tx in transactions]

        while len(hashes) > 1:
            if len(hashes) % 2 == 1:
                hashes.append(hashes[-1])
            hashes = [hashlib.sha256(hashes[i] + hashes[i+1]).digest()
                     for i in range(0, len(hashes), 2)]

        return hashes[0].hex()

    def create_transaction(self, sender: str, operation: str,
                          data: Dict, private_key=None) -> Transaction:
        """Create a signed transaction"""
        tx_id = hashlib.sha256(
            f"{sender}{operation}{time.time()}{random.random()}".encode()
        ).hexdigest()[:16]

        data_str = json.dumps(data, sort_keys=True)
        data_hash = hashlib.sha256(data_str.encode()).hexdigest()

        # Simulate Ed25519 signature
        signature = hashlib.sha256(
            f"{tx_id}{sender}{data_hash}".encode()
        ).hexdigest()

        return Transaction(
            tx_id=tx_id,
            sender=sender,
            operation=operation,
            data_hash=data_hash,
            timestamp=time.time(),
            signature=signature,
            metadata=data
        )

    def mine_block(self, validator: str) -> Block:
        """Simulate block finalization (3s block time)"""
        if validator not in self.validators:
            raise ValueError(f"Invalid validator: {validator}")

        block = Block(
            index=len(self.chain),
            timestamp=time.time(),
            transactions=self.pending_transactions.copy(),
            previous_hash=self.chain[-1].hash,
            merkle_root=self.compute_merkle_root(self.pending_transactions),
            hash="",
            validator=validator
        )
        block.hash = block.compute_hash()
        self.chain.append(block)
        self.pending_transactions = []
        return block

    def verify_chain(self) -> Tuple[bool, List[str]]:
        """Verify entire chain integrity"""
        issues = []
        for i in range(1, len(self.chain)):
            current = self.chain[i]
            previous = self.chain[i-1]

            if current.previous_hash != previous.hash:
                issues.append(f"Block {i}: Previous hash mismatch")
            if current.hash != current.compute_hash():
                issues.append(f"Block {i}: Hash mismatch")
            if current.merkle_root != self.compute_merkle_root(current.transactions):
                issues.append(f"Block {i}: Merkle root mismatch")

        return len(issues) == 0, issues

    def get_tamper_probability(self, byzantine_nodes: int = 7) -> float:
        """
        Calculate probability of undetected tampering
        With 25 validators, 7 Byzantine nodes (28%), threshold 3-of-5
        """
        n = 25
        f = byzantine_nodes
        t = self.threshold

        # Probability of compromising threshold signatures
        # Hypergeometric distribution approximation
        if f < t:
            return 10**-12  # Negligible

        # Simplified: probability of controlling t-of-5 in any group
        p_compromise = math.comb(f, t) / math.comb(n, t)
        return p_compromise * 10**-9  # Additional crypto security

# Initialize blockchain
print("\n" + "=" * 70)
print("PART 1: PERMISSIONED BLOCKCHAIN INITIALIZATION")
print("=" * 70)
blockchain = PermissionedBlockchain()
print(f"Initialized permissioned blockchain with {len(blockchain.validators)} validators")
print(f"Threshold signature scheme: {blockchain.threshold}-of-5")
print(f"Genesis block hash: {blockchain.chain[0].hash[:16]}...")
print(f"Probability of undetected tamper (7 Byzantine nodes): < {blockchain.get_tamper_probability():.2e}")


PART 1: PERMISSIONED BLOCKCHAIN INITIALIZATION
Initialized permissioned blockchain with 25 validators
Threshold signature scheme: 3-of-5
Genesis block hash: 1a509dc572c8e0b0...
Probability of undetected tamper (7 Byzantine nodes): < 1.52e-11
